[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Boolean Masks


## What you will be able to do

Make a mask from a comparison, filter the readings with it, count hours with `.sum()`, combine
conditions with `&`, `|` and `~`, and change values through a mask, all without writing a loop.

## The idea

### The problem

The **Indexing and Slicing** notebook picked readings by where they are, a station, an hour, a range.
Many questions pick them by what they are. How many hours was each station below freezing? What was
the mean of only those hours? Which hours were between 0 and 5 degrees, and when was Oslo first
warmer than 19? None of these has a fixed position to slice at, because the answer depends on the
values.

With lists, each of these questions is a loop with an `if` inside it, either a counter that goes up
when a reading is below zero, or a new list that collects the readings that pass. The condition is
buried inside the loop, and a second condition means a second `if`, or an `and` that has to be read
carefully.

### What a boolean mask is

> A **boolean mask** is an array of `True` and `False` values with the same shape as the array it
> describes, usually made by a comparison such as `readings < 0`. Indexing an array with a mask,
> `readings[mask]`, keeps the values where the mask is `True`, as a new array.

### Why it works that way

A comparison between an array and a number works value by value, like arithmetic, so `readings[0] <
0` is 8,760 answers, one per hour. The answers are an array in their own right, of dtype `bool`, and
because `True` counts as 1 and `False` as 0, `.sum()` on a mask counts the hours that passed and
`.mean()` gives the fraction of them.

Conditions combine with `&` for and, `|` for or and `~` for not, each working value by value. Python's
own `and`, `or` and `not` cannot do that. They need one `True` or `False` for the whole array, and an
array of 8,760 answers is not one answer, so NumPy raises instead of guessing. `&` also binds more
tightly than `<`, so each comparison needs its own parentheses.

Filtering returns a copy where slicing returns a view, as the **Fancy Indexing** notebook shows, and
from a grid it returns a single run of values, because the values that pass do not form rows of
equal length. And `nan` fails every comparison, `nan < 0` and `nan >= 0` alike, so a mask quietly
leaves out the hours a station was silent. `np.isnan` is the mask that finds them.

### Where this shows up

Filtering rows of a table, `table[table["celsius"] < 0]`, is a boolean mask, and the **Pandas, Deep
Dive** guide filters tables that way. Cleaning data, replacing a sentinel value or capping an
outlier, is a mask on the left of `=`. In an image, the pixels brighter than a threshold are a mask
too.

### What this notebook covers

- a comparison that makes a mask, and its dtype `bool`
- filtering with a mask, and counting and finding fractions with `.sum()` and `.mean()`
- two conditions with `&` and `|`, and the opposite with `~`
- the positions where a mask is `True`, with `np.flatnonzero`
- a mask along one dimension, to take a season from every station at once
- changing values through a mask, and filling `nan` with `np.isnan`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

celsius = np.array([2.5, -1.0, 0.5, -3.0])
print(celsius < 0)
print(celsius[celsius < 0])
print((celsius < 0).sum())
```

```
[False  True False  True]
[-1. -3.]
2
```

The comparison gave one answer per reading. Used as an index, it kept the two readings below zero,
and summed, it counted them.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### A comparison makes a mask

In [2]:
frost = readings[0] < 0

print(frost.dtype, frost.shape)
print(frost[:6])
print(readings[0, :6])

bool (8760,)
[ True  True  True  True  True  True]
[-3.6 -3.7 -3.7 -3.5 -3.1 -2.5]


One `True` or `False` for each of Bergen's 8,760 hours. The first six hours of the year were all
below zero.

### Filtering with a mask

Used as an index, the mask keeps the readings where it holds `True`:

In [3]:
cold = readings[0][frost]

print(cold.size)
print(round(cold.mean(), 2), cold.min())

1220
-1.96 -4.8


1,220 of Bergen's hours were below freezing, at a mean of -1.96 degrees, and the coldest was -4.8.

### Counting and fractions

`True` counts as 1, so the sum of a mask is a count and its mean a fraction. A comparison on the
whole grid makes a mask of the same shape, and `axis=1` sums each station's row:

In [4]:
below_zero = readings < 0

for name, hours, share in zip(STATIONS, below_zero.sum(axis=1), below_zero.mean(axis=1)):
    print(f"{name:8} {hours:5} hours  {share:.1%}")

Bergen    1220 hours  13.9%
Oslo      1848 hours  21.1%
Svalbard  5871 hours  67.0%
Tromso    3203 hours  36.6%


Svalbard spent two thirds of the year below freezing, and Bergen about a seventh. The last of the
Common errors below, on a count that leaves out the missing hours, shows what Svalbard's silent day
did to its count.

### Combining conditions with & and |

`&` keeps the hours where both conditions hold, and `|` the hours where either does. Each comparison
goes in its own parentheses:

In [5]:
mild = (readings[0] >= 0) & (readings[0] < 5)
extremes = (readings[0] < 0) | (readings[0] > 18)

print(mild.sum(), "hours from 0 up to 5 degrees")
print(extremes.sum(), "hours below 0 or above 18")

2161 hours from 0 up to 5 degrees
1779 hours below 0 or above 18


### The opposite with ~

`~` turns every `True` into `False` and every `False` into `True`:

In [6]:
print((~frost).sum())
print(frost.sum() + (~frost).sum())

7540
8760


Bergen had 7,540 hours at or above zero, which with the 1,220 below it make up the 8,760 hours of
its year.

### Where a mask is True

`np.flatnonzero` gives the positions where a mask holds `True`, and a position is an hour of the year:

In [7]:
warm = np.flatnonzero(readings[1] > 19)

print(warm.size, warm)
print(START + timedelta(hours=int(warm[0])))

6 [4624 4720 4743 4839 4862 5032]
2025-07-12 16:00:00


Oslo was warmer than 19 degrees in six hours of the year, the first of them at 16:00 on 12 July.

### A mask along one dimension

A mask need not cover the whole grid. One made from the hours alone, `True` for June to August, picks
those columns at every station, and the result keeps its rows:

In [8]:
hours = np.arange(HOURS)
summer = (hours >= hour_of_year(6, 1)) & (hours < hour_of_year(9, 1))

print(summer.sum())
print(readings[:, summer].shape)
print(np.nanmean(readings[:, summer], axis=1).round(2))

2208
(4, 2208)
[16.08 14.58  3.58 11.58]


The summer is 2,208 hours, or 92 days, and the result holds them for all four stations. A mask on
the whole grid gives a single run of values instead, because the values that pass do not form rows
of equal length:

In [9]:
print(readings[readings > 18].shape)

(696,)


The result is 696 readings above 18 degrees from all four stations together, with nothing left to
say which station each came from.

### Changing values through a mask

A mask on the left of `=` replaces the values where it holds `True`. Here a copy of Svalbard's
readings has its missing hours filled with the station's yearly mean from `STATIONS`, using
`np.isnan` as the mask:

In [10]:
svalbard = readings[2].copy()
svalbard[np.isnan(svalbard)] = STATIONS["Svalbard"]

print(np.isnan(readings[2]).sum(), np.isnan(svalbard).sum())
print(svalbard[hour_of_year(3, 2):hour_of_year(3, 2) + 3])

24 0
[-4.5 -4.5 -4.5]


## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/06-boolean-masks-solutions.ipynb).

**1.** Count Tromso's hours above 10 degrees, row 3.

In [11]:
# your code here

**2.** Print the mean of Oslo's readings that were below freezing, rounded to two places.

In [12]:
# your code here

**3.** Count Bergen's hours from 10 degrees up to 15, using `&`.

In [13]:
# your code here

**4.** Make a mask of the hours in December from `np.arange(HOURS)` and `hour_of_year`, and print Svalbard's December mean, rounded to two places.

In [14]:
# your code here

**5.** Copy Oslo's readings, replace every reading above 18 with 18, and print the largest value before and after.

In [15]:
# your code here

## Common errors

### ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [16]:
if readings[0] < 0:
    print("Bergen froze")

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

`if` needs one answer, and `readings[0] < 0` is 8,760 of them. NumPy will not guess which one you
meant, so the message names the two ways to make one answer: `.any()`, true if any hour was below
zero, and `.all()`, true if every hour was. To ask whether any hour was below zero, write `if (readings[0] < 0).any():`.
Python's `and` asks the same question of each side, and raises the same error:

In [17]:
(readings[0] >= 0) and (readings[0] < 5)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

Use `&` between two masks, and `|` in place of `or`.

### TypeError: ufunc 'bitwise_and' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [18]:
readings[0] >= 0 & readings[0] < 5

TypeError: ufunc 'bitwise_and' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

`&` binds more tightly than `>=` and `<`, so Python read the middle first, `0 & readings[0]`, a
bitwise and between a whole number and an array of decimals, which has no meaning. The message names
`bitwise_and`, the operation `&` stands for. Put each comparison in parentheses:
`(readings[0] >= 0) & (readings[0] < 5)`.

### IndexError: boolean index did not match indexed array along axis 0; size of axis is 4 but size of corresponding boolean axis is 8760

In [19]:
readings[frost]

IndexError: boolean index did not match indexed array along axis 0; size of axis is 4 but size of corresponding boolean axis is 8760

`frost` was made from Bergen's row, 8,760 long, and was used on the whole grid, whose first dimension
is the four stations. A mask has to match the dimension it indexes. Use it on the row it came from,
`readings[0][frost]`, or along the hours, `readings[:, frost]`, to pick the same hours at every
station.

### No error, and a count that leaves out the missing hours

In [20]:
below = (readings[2] < 0).sum()
at_or_above = (readings[2] >= 0).sum()

print(below, at_or_above, below + at_or_above)
print(np.isnan(readings[2]).sum())

5871 2865 8736
24


Every hour is either below zero or not, yet the two counts add up to 8,736, which is 24 short of the
8,760 hours in the year. The 24 `nan` hours failed both comparisons, as `nan` fails every
comparison, so they fell out of each count with no warning, and Svalbard's 67% below freezing is a
share of 8,760 that includes 24 hours with no reading. Count the missing hours with `np.isnan`, and
leave them out of the total before you divide: `(readings[2] < 0).sum() /
(~np.isnan(readings[2])).sum()`.

## Recap

- Make a mask with a comparison, and filter with `readings[mask]`
- Count with `mask.sum()` and find a fraction with `mask.mean()`
- Combine conditions with `&`, `|` and `~`, each comparison in its own parentheses
- Use a mask along one dimension to keep rows, and know that a mask on the whole grid gives one run
- Replace values with a mask on the left of `=`, and find `nan` with `np.isnan`

## What is next

The **Fancy Indexing** notebook picks values with an array of positions, puts them in an order of
your choosing, and shows why the result is always a copy.

---

&#8592; **Previous:** [Indexing and Slicing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/05-indexing-and-slicing.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
